## 3. RK45 numerical references (partial Task 3)

The clean numerical-reference milestone is implemented. **SCAFFOLD_ONLY: full threshold proofs, basin maps, extinction-time sweeps and remaining Task 3 work are pending.** Devam's detailed explanation review is also pending.

For known parameters, Model A obeys $N'=rN(1-N/K)(N/A-1)-H$ and Model B replaces $-H$ with $-hN$. The reference solver uses $x=N/K$, $\tau=rt$, $a=A/K$, $q=H/(rK)$ and $e=h/r$, then exports **dimensional** $t=\tau/r,N=Kx$.

SciPy RK45 is the Dormand–Prince 5(4) pair: shared trial slopes yield fourth- and fifth-order estimates; their difference estimates local error and controls adaptive step size. The fifth-order state advances the trajectory. Our dimensionless tolerances are rtol=$10^{-9}$, atol=$10^{-11}$, maximum step 0.1. Requested output times do not set internal steps. See [SciPy solve_ivp documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html).

For A with positive H, zero is **not** an equilibrium. We stop at its first downward zero event and retain the endpoint, with no artificial zero tail. B approaches zero asymptotically from positive starts; finite-horizon smallness is not exact extinction.

The following command regenerates six small clean A/B examples and independently validates them using the expanded dimensional polynomial with DOP853 and tighter RK45. For a strictly declining, reaching-zero A path, separation also gives $t_{ext}=\int_0^{N_0}dN/(H-g(N))$. Quadrature is rejected if a possible equilibrium pole lies on the path. This is a numerical safety check, not the full basin proof.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
run = subprocess.run(
    [
        sys.executable,
        str(ROOT / "experiments/wp03/generate_reference.py"),
        "--config",
        str(ROOT / "configs/wp03_reference.json"),
        "--output",
        str(ROOT / "results/wp03-notebook"),
    ],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(run.stdout.strip())

### Read and interpret the numerical checks

Each case has 101 clean synthetic points in arbitrary model_time_unit/population_unit units; these are not real observations or final inverse benchmarks. Every sampled difference must be <=5e-8*K and event-time differences <=1e-7. These are numerical acceptance targets; they do not guarantee PINN performance. The CSV below retains each case instead of hiding errors in an aggregate. The displayed figure is regenerated by the preceding command, not loaded from an unexplained result.

WP05 must make separate noisy observation files, freeze train/validation/test splits and prevent truth leakage. The known-K design and initial populations are in conditions.json; truth.json and the full manifest/config are evaluation metadata, not optimiser inputs. No PINN is trained here.

In [ ]:
import csv

from IPython.display import Image, display

output = ROOT / "results/wp03-notebook"
with (output / "validation.csv").open() as stream:
    evidence = list(csv.DictReader(stream))
assert all(row["status"] == "passed" for row in evidence)
for row in evidence:
    print(
        row["trajectory_id"],
        "max population difference:",
        row["max_abs_population_error_DOP853"],
        "extinction time:",
        row["extinction_time"] or "no event observed",
    )
display(Image(filename=str(output / "wp03_rk45_reference.png")))

### Scope, provenance and next work

The six numerical checks support this reference-data milestone only. A finite horizon is not proof of permanent survival; near-fold cases are selected examples. See docs/WP03_RK45_HANDOFF.md for code/data links, custom sampling, file hashes, measured comparisons and exact regeneration commands. Manifest timestamps/runtime and figure metadata vary; clean CSV/conditions/truth bytes repeat in the same dependency environment.

Full collapse-threshold derivations, basin/extinction maps, optional sensitivity/MSY and final submission reproduction remain pending. WP05 performs parameter/noise/held-out studies, and WP04 supplies the shared PINN engine. AI assistance: OpenAI ChatGPT/Codex implemented this numerical milestone; human scientific review and Devam's own understanding check remain pending.